In [ ]:
import sys
import os
import shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    LOCAL_DATA_ROOT = '/content/AA-STAL_DATA_ROOT'
    print(f"Drive root: {DRIVE_ROOT}")
except ImportError:
    # Fallback for local execution
    DRIVE_ROOT = '.'
    LOCAL_DATA_ROOT = os.path.abspath('DATA_ROOT')
    print("Local execution")

DRIVE_DATA_ROOT = os.path.join(DRIVE_ROOT, 'DATA_ROOT')
os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)
os.chdir(LOCAL_DATA_ROOT)
print(f"Working directory (local): {os.getcwd()}")

def copy_directory(source_dir, destination_dir):
    """Copy a directory tree, merging it with an existing destination."""
    if os.path.abspath(source_dir) == os.path.abspath(destination_dir):
        print(f'Already local: {source_dir}')
        return
    if not os.path.isdir(source_dir):
        raise FileNotFoundError(f'Source directory not found: {source_dir}')
    print(f'Copying {source_dir} -> {destination_dir}')
    shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)

def sync_results_to_drive(*directory_names):
    """Persist completed local result directories to Google Drive."""
    for directory_name in directory_names:
        local_dir = os.path.join(LOCAL_DATA_ROOT, directory_name)
        drive_dir = os.path.join(DRIVE_DATA_ROOT, directory_name)
        copy_directory(local_dir, drive_dir)
    print('Results synchronized to Google Drive.')

In [ ]:
!pip install scenedetect
!pip install tqdm

In [ ]:
import os
import glob
import subprocess
import tempfile
from tqdm import tqdm
from scenedetect import detect, ContentDetector

def get_video_duration(video_path):
    """Gets video durantion in seconds using ffprobe."""
    cmd = [
        "ffprobe",
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        video_path
    ]
    try:
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
        return float(result.stdout.strip())
    except Exception:
        return 0.0

def crop_video_segments(video_path, segment_jobs):
    """Encode all contiguous segments of one video in a single FFmpeg run."""
    if not segment_jobs:
        return

    for previous, current in zip(segment_jobs, segment_jobs[1:]):
        if abs(previous[1] - current[0]) > 1e-3:
            raise ValueError('Video segments must be contiguous and ordered.')

    input_start = segment_jobs[0][0]
    total_duration = segment_jobs[-1][1] - input_start
    split_times = [end - input_start for _, end, _ in segment_jobs[:-1]]

    with tempfile.TemporaryDirectory(prefix='ffmpeg_segments_', dir=os.path.dirname(segment_jobs[0][2])) as temp_dir:
        cmd = [
            'ffmpeg', '-y', '-loglevel', 'error',
            '-ss', str(input_start), '-i', video_path, '-t', str(total_duration),
            '-map', '0:v:0', '-map', '0:a?',
            '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-preset', 'veryfast', '-crf', '23',
            '-c:a', 'aac', '-b:a', '128k',
        ]

        if split_times:
            times = ','.join(f'{time:.6f}' for time in split_times)
            temp_pattern = os.path.join(temp_dir, 'segment_%06d.mp4')
            cmd.extend([
                '-force_key_frames', times,
                '-f', 'segment', '-segment_times', times, '-segment_time_delta', '0.05',
                '-reset_timestamps', '1',
                temp_pattern,
            ])
        else:
            cmd.append(os.path.join(temp_dir, 'segment_000000.mp4'))

        subprocess.run(cmd, check=True)
        generated_segments = sorted(glob.glob(os.path.join(temp_dir, 'segment_*.mp4')))
        if len(generated_segments) != len(segment_jobs):
            raise RuntimeError(
                f'FFmpeg produced {len(generated_segments)} segments; expected {len(segment_jobs)}.'
            )

        for generated_path, (_, _, output_path) in zip(generated_segments, segment_jobs):
            os.replace(generated_path, output_path)

def chunk_into_n(lst, n):
    """Split the list in n chunk."""
    chunk_size = len(lst) // n
    remainder = len(lst) % n

    chunks = []
    start = 0
    for i in range(n):
        end = start + chunk_size + (1 if i < remainder else 0)
        chunks.append(lst[start:end])
        start = end

    return chunks

def process_videos(input_dir, output_dir, chunk_idx=None, chunk_num=None):
    os.makedirs(output_dir, exist_ok=True)

    video_files = glob.glob(os.path.join(input_dir, "*.mp4"))
    video_files.extend(glob.glob(os.path.join(input_dir, "*.avi")))
    video_files.extend(glob.glob(os.path.join(input_dir, "*.mov")))
    video_files.sort()

    if chunk_idx is not None and chunk_num is not None:
        video_chunks = chunk_into_n(video_files, chunk_num)
        if chunk_idx >= len(video_chunks):
            print(f"Error: Chunk {chunk_idx} out of range (tot: {len(video_chunks)})")
            return
        video_files = video_chunks[chunk_idx]
        print(f"Chunk processing {chunk_idx}/{chunk_num-1}: {len(video_files)} video")
    else:
        print(f"Found {len(video_files)} video to process")

    for video_path in tqdm(video_files):
        video_name = os.path.basename(video_path)
        base_name = os.path.splitext(video_name)[0]

        # 1. Scene split
        try:
            raw_scenes = detect(video_path, ContentDetector())
        except Exception as e:
            print(f"Error scenedetect in {video_name}: {e}")
            raw_scenes = []

        scenes_sec = []

        # 2. continuative video vs cut video
        if not raw_scenes:
            # no cut found
            total_duration = get_video_duration(video_path)
            if total_duration > 0:
                scenes_sec = [(0.0, total_duration)]
                print(f"Video {video_name}: Nessun taglio. Trattato come scena singola.")
            else:
                print(f"Video {video_name}: File corrotto o illeggibile.")
                continue
        else:
            scenes_sec = [(s[0].seconds, s[1].seconds) for s in raw_scenes]
            print(f"Video {video_name}: trovate {len(scenes_sec)} scene.")

        # 3. Scene extraction
        valid_scenes = 0
        split_scenes = []
        for i, (start_sec, end_sec) in enumerate(scenes_sec):
            duration = end_sec - start_sec
            if duration > 10.0:
                # if the scene exceeds 10s it is splitted in 10s segments until it remains a < 10s part.
                cur_start = start_sec
                part = 1
                while end_sec - cur_start >= 10.0:
                    split_scenes.append((cur_start, cur_start + 10.0, i + 1, part))
                    cur_start += 10.0
                    part += 1
                if end_sec - cur_start > 0:
                    split_scenes.append((cur_start, end_sec, i + 1, part))
                total_parts = part - (0 if end_sec - cur_start > 0 else 1)
                print(f"Video {video_name}: scena {i+1} di {duration:.2f}s divisa in {total_parts} blocchi da 10s e eventuale resto")
            else:
                split_scenes.append((start_sec, end_sec, i + 1, None))

        segment_jobs = []
        for start_sec, end_sec, scene_idx, part_idx in split_scenes:
            duration = end_sec - start_sec
            if duration <= 0:
                continue
            valid_scenes += 1
            if part_idx is None:
                output_name = f"{base_name}_scene{scene_idx}.mp4"
            else:
                output_name = f"{base_name}_scene{scene_idx}_part{part_idx}.mp4"
            output_path = os.path.join(output_dir, output_name)
            segment_jobs.append((start_sec, end_sec, output_path))

        crop_video_segments(video_path, segment_jobs)

        print(f"  -> Salvate {valid_scenes} clip valide su {len(split_scenes)} totali.")

if __name__ == "__main__":

    drive_in_dir = os.path.join(DRIVE_DATA_ROOT, 'Videos')
    in_dir = os.path.join(LOCAL_DATA_ROOT, 'Videos')
    out_dir = os.path.join(LOCAL_DATA_ROOT, 'Videos_crop')
    chunk_idx=None
    chunk_num=None

    # Stage the source videos on Colab's local disk. All heavy processing
    # below reads from and writes to /content, not to the Drive mount.
    copy_directory(drive_in_dir, in_dir)

    process_videos(in_dir, out_dir, chunk_idx, chunk_num)
    print(f"Local crop completed: {out_dir}")

In [ ]:
import os
import glob
import sys
import subprocess
from shutil import which
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable):
        return iterable

def chunk_into_n(lst, n):
    """Divide list into n chunks as evenly as possible"""
    chunk_size = len(lst) // n
    remainder = len(lst) % n

    chunks = []
    start = 0
    for i in range(n):
        # Add one extra item to the first 'remainder' chunks
        end = start + chunk_size + (1 if i < remainder else 0)
        chunks.append(lst[start:end])
        start = end

    return chunks

def _resolve_ffmpeg_binary() -> str:
    # 1) Respect env override
    env_bin = os.environ.get('FFMPEG')
    if env_bin and os.path.isfile(env_bin) and os.access(env_bin, os.X_OK):
        return env_bin
    # 2) Use system PATH
    sys_bin = which('ffmpeg')
    if sys_bin:
        return sys_bin
    # 3) Try alongside the current Python executable
    py_dir = os.path.dirname(sys.executable)
    candidate = os.path.join(py_dir, 'ffmpeg')
    if os.path.isfile(candidate) and os.access(candidate, os.X_OK):
        return candidate
    # 4) Try known conda env path
    known_candidate = '/data/vision/torralba/selfmanaged/isola/u/yulu/miniconda3/envs/fm/bin/ffmpeg'
    if os.path.isfile(known_candidate) and os.access(known_candidate, os.X_OK):
        return known_candidate
    return 'ffmpeg'  # fallback; will likely fail with 'not found'

def _resolve_ffprobe_binary() -> str:
    env_bin = os.environ.get('FFPROBE')
    if env_bin and os.path.isfile(env_bin) and os.access(env_bin, os.X_OK):
        return env_bin
    sys_bin = which('ffprobe')
    if sys_bin:
        return sys_bin
    ffmpeg_dir = os.path.dirname(_resolve_ffmpeg_binary())
    candidate = os.path.join(ffmpeg_dir, 'ffprobe')
    if os.path.isfile(candidate) and os.access(candidate, os.X_OK):
        return candidate
    return 'ffprobe'

def has_video_stream(video_path):
    """Check if the video file contains a video stream."""
    cmd = [
        _resolve_ffprobe_binary(), '-v', 'error',
        '-select_streams', 'v:0',
        '-show_entries', 'stream=index',
        '-of', 'csv=p=0',
        video_path,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0 and bool(result.stdout.strip())

def decode_video(video_path, save_dir, video_name=''):
    os.makedirs(save_dir, exist_ok=True)

    # Check if the file has a video stream
    if not has_video_stream(video_path):
        print(f"Warning: {video_path} does not contain a video stream. Skipping...")
        # Create a marker file to indicate this is an audio-only file
        with open(os.path.join(save_dir, 'audio_only.txt'), 'w') as f:
            f.write(f"This file ({video_path}) contains only audio, no video stream to extract frames from.")
        return

    # Use more robust ffmpeg parameters to handle videos with timestamp issues
    ffmpeg_bin = _resolve_ffmpeg_binary()
    cmd = f"{ffmpeg_bin} -loglevel error -fflags +genpts -i {video_path} -r 30 -f image2 -qscale:v 1 '{save_dir}/%10d.jpg'"
    result = subprocess.run(cmd, shell=True)
    first_frame = os.path.join(save_dir, f'{1:10d}.jpg')
    extraction_succeeded = result.returncode == 0 and os.path.isfile(first_frame)
    print(f"{video_path}, extraction succeeded={extraction_succeeded}")

    # If no frames extracted, try alternative approach
    if not extraction_succeeded:
        print(f"Warning: No frames extracted with standard method, trying alternative approach...")
        # Clear directory first
        for f in os.listdir(save_dir):
            os.remove(os.path.join(save_dir, f))
        # Try with video filter to force frame extraction
        cmd_alt = f"{ffmpeg_bin} -loglevel error -fflags +genpts -i {video_path} -vf 'fps=30' -qscale:v 1 '{save_dir}/%10d.jpg'"
        result_alt = subprocess.run(cmd_alt, shell=True)
        alternative_succeeded = result_alt.returncode == 0 and os.path.isfile(first_frame)
        print(f"{video_path}, alternative extraction succeeded={alternative_succeeded}")

        # If still no frames, mark as problematic
        if not alternative_succeeded:
            with open(os.path.join(save_dir, 'extraction_failed.txt'), 'w') as f:
                f.write(f"Failed to extract frames from {video_path} using both standard and alternative methods.")


if __name__ == '__main__':

    # parser = argparse.ArgumentParser(description='Decode videos to frames')
    # parser.add_argument('--clip_dir', type=str, default='', help='Folder with input video folder')
    # parser.add_argument('--chunk_idx', type=int, default=None,
    #                     help='Chunk index to process (for parallel processing)')
    # parser.add_argument('--chunk_num', type=int, default=None,
    #                     help='Total number of chunks (for parallel processing)')
    # args = parser.parse_args()

    clip_dir = os.path.join(LOCAL_DATA_ROOT, 'Videos_crop')
    chunk_idx=None
    chunk_num=None

    # Directory for decoded frames
    decode_dir = clip_dir.rstrip('/') + '_decode'

    # Get all video files
    video_paths = glob.glob(f'{clip_dir}/*.mp4')
    video_paths.sort()  # Sort for consistent chunking

    # Handle chunking
    if chunk_idx is not None and chunk_num is not None:
        video_chunks = chunk_into_n(video_paths, chunk_num)
        if chunk_idx >= len(video_chunks):
            print(f"Chunk {chunk_idx} is out of range (total chunks: {len(video_chunks)})")
            exit(0)
        video_paths = video_chunks[chunk_idx]
        print(f"Processing chunk {chunk_idx}/{chunk_num-1} with {len(video_paths)} videos")
    else:
        print(f"Processing all {len(video_paths)} videos")

    for video_path in tqdm(video_paths):
        video_name = video_path.split('/')[-1][:-4]
        save_dir = os.path.join(decode_dir, video_name)
        decode_video(video_path, save_dir)

    # Drive is used only for persistence after all local processing has finished.
    sync_results_to_drive('Videos_crop', 'Videos_crop_decode')